# 4 - Tools (Ferramentas) no LangChain

**Tools** são funções Python que o LLM pode decidir chamar durante sua execução.
Em vez de apenas gerar texto, o modelo pode invocar ferramentas para buscar dados reais,
fazer cálculos, consultar APIs, e muito mais.

É a base dos **agentes**: o LLM raciocina sobre qual ferramenta usar, com quais argumentos,
e incorpora o resultado na resposta final.

```
Pergunta do usuário
        │
        ▼
       LLM  ──► "preciso calcular distância"  ──►  calcular_distancia(origem, destino)
        │                                                    │
        └────────────────────────────────────────────────────┘
                         resultado da tool
        │
        ▼
  Resposta final fundamentada no resultado real
```

### O que mudou na API moderna

| Antigo (deprecado) | Moderno |
|--------------------|--------|
| `from langchain.pydantic_v1 import BaseModel, Field` | `from pydantic import BaseModel, Field` |
| `from langchain.agents import tool` | `from langchain_core.tools import tool` |

> O decorator `@tool` e o parâmetro `args_schema` continuam funcionando da mesma forma —
> apenas os imports mudam para os pacotes corretos.

### Pacotes necessários

```bash
pip install langchain-core pydantic
```

## 1. Criando uma Tool com `@tool`

A forma mais simples de criar uma tool é usar o decorator `@tool` em uma função Python.
O LangChain usa três informações da função para expô-la ao LLM:

| Informação | Origem | Uso |
|------------|--------|-----|
| **Nome** | nome da função | identifica a tool |
| **Descrição** | docstring da função | explica ao LLM quando usá-la |
| **Schema dos args** | `args_schema` (Pydantic) | define os parâmetros esperados |

> **`args_schema`**: ao passar um modelo Pydantic com `Field(description=...)`,
> o LLM recebe instruções precisas sobre o que cada argumento representa.
> Isso melhora muito a qualidade das chamadas automáticas.

In [2]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class CalcularDistanciaArgs(BaseModel):
    cidade_origem:  str = Field(
        description="Cidade de origem da viagem",
        examples=["São Paulo", "Rio de Janeiro"]
    )
    cidade_destino: str = Field(
        description="Cidade de destino da viagem",
        examples=["Porto Alegre", "Curitiba"]
    )

@tool(args_schema=CalcularDistanciaArgs)
def calcular_distancia(cidade_origem: str, cidade_destino: str) -> str:
    """Calcula a distância em km entre duas cidades brasileiras."""
    # Em produção, aqui chamaria uma API de mapas (Google Maps, etc.)
    return f"A distância entre {cidade_origem} e {cidade_destino} é de 300km"

# Inspecionando a tool criada
print(f"Nome:      {calcular_distancia.name}")
print(f"Descrição: {calcular_distancia.description}")
print(f"Schema:    {calcular_distancia.args}")

Nome:      calcular_distancia
Descrição: Calcula a distância em km entre duas cidades brasileiras.
Schema:    {'cidade_origem': {'description': 'Cidade de origem da viagem', 'title': 'Cidade Origem', 'type': 'string'}, 'cidade_destino': {'description': 'Cidade de destino da viagem', 'title': 'Cidade Destino', 'type': 'string'}}


In [3]:
# Chamando a tool diretamente com .invoke()
# .invoke() é o método padrão LCEL — substitui chamar a função diretamente
resultado = calcular_distancia.invoke({
    "cidade_origem": "São Paulo",
    "cidade_destino": "Porto Alegre"
})
print(resultado)

A distância entre São Paulo e Porto Alegre é de 300km


## 2. Tool com operações matemáticas

Tools podem encapsular qualquer lógica Python — cálculos, consultas a banco de dados,
chamadas de API, leitura de arquivos, etc.

O importante é que a **docstring** seja descritiva: é o que o LLM lê para decidir
se deve ou não chamar essa ferramenta.

In [4]:
class OperacaoMatematicaArgs(BaseModel):
    numero1:  float = Field(description="Primeiro número da operação",  examples=[10, 5])
    numero2:  float = Field(description="Segundo número da operação",   examples=[20, 3])
    operacao: str   = Field(
        description="Operação a ser realizada: 'soma', 'subtração', 'multiplicação' ou 'divisão'",
        examples=["soma", "subtração"]
    )

@tool(args_schema=OperacaoMatematicaArgs)
def realizar_calculo(numero1: float, numero2: float, operacao: str) -> float | str:
    """Realiza operações matemáticas básicas: soma, subtração, multiplicação e divisão."""
    if operacao == "soma":
        return numero1 + numero2
    elif operacao == "subtração":
        return numero1 - numero2   # ⚠️ bug corrigido: era numero1 + numero2
    elif operacao == "multiplicação":
        return numero1 * numero2
    elif operacao == "divisão":
        if numero2 == 0:
            return "Erro: divisão por zero"
        return numero1 / numero2
    else:
        return f"Operação inválida: '{operacao}'. Use soma, subtração, multiplicação ou divisão."

realizar_calculo

StructuredTool(name='realizar_calculo', description='Realiza operações matemáticas básicas: soma, subtração, multiplicação e divisão.', args_schema=<class '__main__.OperacaoMatematicaArgs'>, func=<function realizar_calculo at 0x0000024200DDD080>)

In [5]:
# Testando todas as operações
testes = [
    {"numero1": 10, "numero2": 50, "operacao": "soma"},
    {"numero1": 10, "numero2": 50, "operacao": "subtração"},
    {"numero1": 10, "numero2": 50, "operacao": "multiplicação"},
    {"numero1": 10, "numero2": 50, "operacao": "divisão"},
]

for teste in testes:
    resultado = realizar_calculo.invoke(teste)
    print(f"{teste['numero1']} {teste['operacao']} {teste['numero2']} = {resultado}")

10 soma 50 = 60.0
10 subtração 50 = -40.0
10 multiplicação 50 = 500.0
10 divisão 50 = 0.2


## 3. Tool com argumento único

Para tools com apenas um argumento, o schema fica mais simples.
O princípio é o mesmo — a docstring instrui o LLM sobre quando usar a ferramenta.

In [6]:
class InformacoesFilmeArgs(BaseModel):
    titulo_filme: str = Field(
        description="Título do filme a buscar informações",
        examples=["Inception", "The Matrix"]
    )

@tool(args_schema=InformacoesFilmeArgs)
def obter_informacoes_filme(titulo_filme: str) -> str:
    """Retorna informações sobre um filme: ano de lançamento e diretor."""
    # Em produção, consultaria uma API como TMDB ou OMDB
    return f'O filme "{titulo_filme}" foi lançado em 2010, dirigido por Christopher Nolan.'

obter_informacoes_filme

StructuredTool(name='obter_informacoes_filme', description='Retorna informações sobre um filme: ano de lançamento e diretor.', args_schema=<class '__main__.InformacoesFilmeArgs'>, func=<function obter_informacoes_filme at 0x0000024200DDD6C0>)

In [7]:
resultado = obter_informacoes_filme.invoke({"titulo_filme": "Inception"})
print(resultado)

O filme "Inception" foi lançado em 2010, dirigido por Christopher Nolan.


## 4. Inspecionando as tools

Cada tool expõe atributos úteis para inspecionar sua estrutura.
Esses mesmos atributos são usados internamente pelo LangChain para informar o LLM
sobre as ferramentas disponíveis.

In [8]:
tools = [calcular_distancia, realizar_calculo, obter_informacoes_filme]

for t in tools:
    print(f"{'─'*50}")
    print(f"Nome:      {t.name}")
    print(f"Descrição: {t.description}")
    print(f"Args:      {t.args}")

──────────────────────────────────────────────────
Nome:      calcular_distancia
Descrição: Calcula a distância em km entre duas cidades brasileiras.
Args:      {'cidade_origem': {'description': 'Cidade de origem da viagem', 'title': 'Cidade Origem', 'type': 'string'}, 'cidade_destino': {'description': 'Cidade de destino da viagem', 'title': 'Cidade Destino', 'type': 'string'}}
──────────────────────────────────────────────────
Nome:      realizar_calculo
Descrição: Realiza operações matemáticas básicas: soma, subtração, multiplicação e divisão.
Args:      {'numero1': {'description': 'Primeiro número da operação', 'title': 'Numero1', 'type': 'number'}, 'numero2': {'description': 'Segundo número da operação', 'title': 'Numero2', 'type': 'number'}, 'operacao': {'description': "Operação a ser realizada: 'soma', 'subtração', 'multiplicação' ou 'divisão'", 'title': 'Operacao', 'type': 'string'}}
──────────────────────────────────────────────────
Nome:      obter_informacoes_filme
Descrição:

## 5. Vinculando tools ao modelo com `bind_tools()`

Para que o LLM possa **chamar** as tools automaticamente, precisamos vinculá-las ao modelo
com `.bind_tools()`. Isso informa ao modelo quais ferramentas estão disponíveis.

O modelo então decide, baseado na pergunta, se deve chamar uma tool e com quais argumentos.

> **Importante**: `.bind_tools()` apenas vincula as tools ao modelo. A **execução** das tools
> e o ciclo completo de agente são tratados no próximo notebook.

In [9]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

chat = ChatOpenAI(model="gpt-4o-mini")

# Vinculando as tools ao modelo
# O modelo agora sabe que pode chamar qualquer uma dessas ferramentas
chat_com_tools = chat.bind_tools(tools)

print("Tools vinculadas:", [t.name for t in tools])

Tools vinculadas: ['calcular_distancia', 'realizar_calculo', 'obter_informacoes_filme']


In [10]:
# O modelo identifica que precisa usar a tool calcular_distancia
# e retorna uma tool_call com os argumentos preenchidos
resposta = chat_com_tools.invoke("Qual a distância de São Paulo para Curitiba?")

print(f"Conteúdo: '{resposta.content}'")
print(f"\nTool calls: {resposta.tool_calls}")

Conteúdo: ''

Tool calls: [{'name': 'calcular_distancia', 'args': {'cidade_origem': 'São Paulo', 'cidade_destino': 'Curitiba'}, 'id': 'call_6VQXIsngRzcOfpSrxlpeoqb5', 'type': 'tool_call'}]


In [11]:
# Quando a pergunta não requer nenhuma tool, o modelo responde normalmente
resposta_direta = chat_com_tools.invoke("Qual é a capital do Brasil?")

print(f"Conteúdo: '{resposta_direta.content}'")
print(f"Tool calls: {resposta_direta.tool_calls}")  # lista vazia — nenhuma tool chamada

Conteúdo: 'A capital do Brasil é Brasília.'
Tool calls: []


## Resumo

| Conceito | Descrição |
|----------|-----------|
| `@tool` | Decorator que transforma uma função Python em uma LangChain Tool |
| `args_schema` | Pydantic model que define e descreve os argumentos da tool |
| `tool.name` | Nome da tool (igual ao nome da função) |
| `tool.description` | Docstring da função — instrui o LLM sobre quando usar a tool |
| `tool.invoke(dict)` | Executa a tool diretamente com um dicionário de argumentos |
| `.bind_tools(lista)` | Vincula tools ao modelo para que ele possa chamá-las |
| `resposta.tool_calls` | Lista das chamadas de tool que o modelo decidiu fazer |

**Padrão moderno completo:**

```python
from langchain_core.tools import tool   # ← import correto
from pydantic import BaseModel, Field   # ← pydantic v2

class MinhaToolArgs(BaseModel):
    parametro: str = Field(description="O que este parâmetro representa")

@tool(args_schema=MinhaToolArgs)
def minha_tool(parametro: str) -> str:
    """Descrição clara de quando o LLM deve usar esta ferramenta."""
    return f"Resultado para {parametro}"

# Uso direto
minha_tool.invoke({"parametro": "valor"})

# Vinculando ao modelo
chat_com_tools = chat.bind_tools([minha_tool])
```

> **Próximo passo**: criar um **agente** que executa automaticamente o ciclo completo —
> o LLM decide qual tool usar, a tool é executada, o resultado volta ao LLM,
> e o processo se repete até ter uma resposta final.